# 00 - Setup Validation

**Goal:** Validate that the baselines_evaluation framework is set up correctly.

This notebook:
1. Tests imports from main codebase
2. Loads catalog and historical data
3. Initializes environment and reward calculator
4. Verifies value signals computation
5. Explores sample contexts

**Status:** Phase 0 - Setup & Directory Structure

## 1. Setup Python Path

In [65]:
import sys
from pathlib import Path

# Add both baselines_evaluation/src and main src to path
project_root = Path.cwd().parent.parent  # Up to project root
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(project_root / "baselines_evaluation" / "src"))

print(f"Project root: {project_root}")
print(f"Main codebase: {project_root / 'src'}")
print(f"Evaluation code: {project_root / 'baselines_evaluation' / 'src'}")

Project root: /Users/theomaetz/Desktop/python
Main codebase: /Users/theomaetz/Desktop/python/src
Evaluation code: /Users/theomaetz/Desktop/python/baselines_evaluation/src


## 2. Test Imports from Main Codebase

In [66]:
# Standard imports
import pandas as pd
import numpy as np
from datetime import datetime, date
import joblib

# Main codebase imports
from cts_recommender.settings import get_settings
from cts_recommender.io.readers import read_parquet
from cts_recommender.environments.TV_environment import TVProgrammingEnvironment
from cts_recommender.environments.reward import RewardCalculator
from cts_recommender.environments.schemas import Context, Season, Channel, TimeSlot
from cts_recommender.models.audience_regression.audience_ratings_regressor import AudienceRatingsRegressor
from cts_recommender.imitation_learning.IL_constants import PSEUDO_REWARD_WEIGHTS
from cts_recommender.features.catalog_schema import CATALOG_DTYPES, HISTORICAL_PROGRAMMING_DTYPES, enforce_dtypes

print("✅ All imports successful!")

✅ All imports successful!


## 3. Load Configuration and Data

In [67]:
# Get settings
cfg = get_settings()

print(f"Environment: {cfg.env}")
print(f"Project root: {cfg.project_root}")
print(f"Data root: {cfg.data_root}")
print(f"Processed dir: {cfg.processed_dir}")
print(f"Models dir: {cfg.models_dir}")
print(f"Random seed: {cfg.random_seed}")

Environment: dev
Project root: /Users/theomaetz/Desktop/python/RTS_curator_recommendation_system/RTS-curator-recommendation-system
Data root: data
Processed dir: data/processed
Models dir: data/models
Random seed: 42


In [68]:
# Load WhatsOn catalog
catalog_path = cfg.processed_dir / "whatson" / "whatson_catalog.parquet"
catalog = read_parquet(catalog_path)

# Enforce data types to ensure datetime columns are properly typed
catalog = enforce_dtypes(catalog, CATALOG_DTYPES)

print(f"Catalog shape: {catalog.shape}")
print(f"Catalog columns: {len(catalog.columns)} columns")
print(f"\nDate column dtypes:")
print(f"  tv_rights_start: {catalog['tv_rights_start'].dtype}")
print(f"  tv_rights_end: {catalog['tv_rights_end'].dtype}")
print(f"\nFirst few rows:")
catalog.head()

Catalog shape: (13682, 46)
Catalog columns: 46 columns

Date column dtypes:
  tv_rights_start: datetime64[ns]
  tv_rights_end: datetime64[ns]

First few rows:


,actors,adult,available_broadcasts,best_title,class_key,collection,consumed_broadcasts,department,director,duration,...,tv_rights_end,tv_rights_start,valid_tv_rights_count,vote_average,missing_release_date,missing_tmdb,is_movie,movie_age,duration_min,times_shown
catalog_id,,,,,,,,,,,,,,,,,,,,,
628,"Tom Cruise, Brad Pitt, Stephen Rea, Antonio Ba...",False,0,Interview with the vampire,71 - Films de cinéma,Film,1,ROUGE,Neil Jordan,01:57:05,...,2026-02-28,2024-03-01,1,7.382,False,0,1,30,117.0,0
820,"Kevin Costner, Sissy Spacek, Joe Pesci, Tommy ...",False,0,JFK,71 - Films de cinéma,Film,1,JAUNE,Oliver Stone,03:17:18,...,2026-04-14,2024-04-15,1,7.604,False,0,1,33,197.0,0
348,"Sigourney Weaver, Tom Skeritt, Harry Dean Stan...",False,1,Alien,71 - Films de cinéma,Film,1,ROUGE,Ridley Scott,01:51:37,...,2027-02-28,2024-03-01,1,8.165,False,0,1,46,111.0,0
9679,"Nicolas Cage, Angelina Jolie, Robert Duvall, G...",False,0,Gone In Sixty Seconds,71 - Films de cinéma,Film,2,JAUNE,Dominic Sena,01:52:43,...,2023-05-31,2020-06-01,0,6.447,False,0,1,25,112.0,0
9552,"Ellen Burstyn, Linda Blair, Max Von Sydow, Jas...",False,0,The Exorcist,71 - Films de cinéma,Film,1,ROUGE,William Friedkin,02:06:33,...,2025-01-31,2023-02-01,0,7.700,False,0,1,51,126.0,0


In [69]:
# Load historical programming
historical_path = cfg.processed_dir / "programming" / "historical_programming.parquet"
historical = read_parquet(historical_path)

# Enforce data types to ensure datetime columns are properly typed
historical = enforce_dtypes(historical, HISTORICAL_PROGRAMMING_DTYPES)

print(f"Historical programming shape: {historical.shape}")
print(f"Date range: {historical['date'].min()} to {historical['date'].max()}")
print(f"Channels: {historical['channel'].unique().tolist()}")
print(f"\nDate column dtype: {historical['date'].dtype}")
print(f"\nFirst few rows:")
historical.head()

Historical programming shape: (3338, 16)
Date range: 2024-01-01 00:00:00 to 2025-02-22 00:00:00
Channels: ['M6_T_PL', 'France 3', 'TF1_T_PL', 'RTS 1', 'RTS 2', 'France 2']

Date column dtype: datetime64[ns]

First few rows:


,catalog_id,date,start_time,channel,title,processed_title,duration_min,rt_m,pdm,hour,weekday,is_weekend,season,public_holiday,tmdb_id,missing_tmdb_id
0,<NA>,2024-01-01,08:02:38,M6_T_PL,EN FAMILLE,En famille,170.750000,7.0,9.4,8,0,False,winter,True,555379,False
1,<NA>,2024-01-01,08:12:19,France 3,SCOOBY-DOO ET LE FANTOME GOURMAND,Scooby-Doo! et le fantôme gourmand,74.116667,0.4,0.7,8,0,False,winter,True,533592,False
2,<NA>,2024-01-01,08:40:56,TF1_T_PL,MOI MOCHE ET MECHANT 2,Moi moche et Méchant & Moi moche et méchant 2 ...,93.766667,4.6,6.5,8,0,False,winter,True,677124,False
3,324852,2024-01-01,10:16:07,TF1_T_PL,MOI MOCHE ET MECHANT 3,"Moi, moche et méchant 3",83.716667,15.4,10.7,10,0,False,winter,True,324852,False
4,83099,2024-01-01,13:57:27,France 3,FORT BRAVO,Fort Bravo,94.883333,7.2,2.9,13,0,False,winter,True,83099,False


In [70]:
# Load audience ratings model
audience_model_path = cfg.models_dir / "audience_ratings_model.joblib"
audience_model = AudienceRatingsRegressor()
audience_model.load_model(audience_model_path)

print(f"Audience model type: {type(audience_model)}")
print(f"Is trained: {audience_model.is_trained}")
print(f"Number of features: {len(audience_model.feature_names_)}")
print("✅ Audience model loaded successfully!")

Audience model type: <class 'cts_recommender.models.audience_regression.audience_ratings_regressor.AudienceRatingsRegressor'>
Is trained: True
Number of features: 71
✅ Audience model loaded successfully!


## 4. Initialize Environment and Reward Calculator

In [71]:
# Initialize environment
env = TVProgrammingEnvironment(
    catalog_df=catalog,
    historical_programming_df=historical,
    audience_model=audience_model
)

print("✅ Environment initialized!")
print(f"Catalog size: {len(env.catalog_df)}")
print(f"Context feature cache: {len(env.context_features_cache)} entries")
print(f"Memory size: {env.memory_size}")

✅ Environment initialized!
Catalog size: 13682
Context feature cache: 0 entries
Memory size: 100


In [72]:
# Reward calculator is initialized within the environment
# Access it via env.reward
reward_calc = env.reward

print("✅ Reward calculator accessed from environment!")
print(f"Reward calculator type: {type(reward_calc)}")

✅ Reward calculator accessed from environment!
Reward calculator type: <class 'cts_recommender.environments.reward.RewardCalculator'>


## 5. Test Context Creation and Features

In [73]:
# Create a sample context manually (Saturday evening on RTS 1)
sample_datetime = pd.Timestamp("2024-03-02 20:40:00")  # Saturday

# Extract context components
hour = sample_datetime.hour
day_of_week = sample_datetime.dayofweek  # 0=Monday, 6=Sunday
month = sample_datetime.month

# Map month to season
if month in [3, 4, 5]:
    season = Season.SPRING
elif month in [6, 7, 8]:
    season = Season.SUMMER
elif month in [9, 10, 11]:
    season = Season.AUTUMN
else:
    season = Season.WINTER

# Map channel
channel = Channel.RTS1

# Create Context object
context = Context(
    hour=hour,
    day_of_week=day_of_week,
    month=month,
    season=season,
    channel=channel
)

# Get context features
context_features, cache_key = env.get_context_features(context)

print(f"Context: {context}")
print(f"Context features shape: {context_features.shape}")
print(f"Context features: {context_features}")
print(f"Cache key: {cache_key}")

Context: Context(hour=20, day_of_week=5, month=3, season=<Season.SPRING: 'spring'>, channel=<Channel.RTS1: 'RTS 1'>)
Context features shape: (18,)
Context features: [1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1. 1. 0. 0. 0. 1. 0.]
Cache key: (20, 5, 3, 'spring', 'RTS 1')


## 6. Test Movie Features and Value Signals

In [74]:
# Get a sample movie from catalog
sample_catalog_id = catalog.index[0]
sample_movie = catalog.loc[sample_catalog_id]

print(f"Sample movie: {sample_movie['title']}")
print(f"Catalog ID: {sample_catalog_id}")

Sample movie: Entretien avec un vampire
Catalog ID: 628


In [75]:
# Get movie features
movie_features = env.get_movie_features(sample_catalog_id)

print(f"Movie features shape: {movie_features.shape}")
print(f"Movie features (first 10): {movie_features[:10]}")

Movie features shape: (24,)
Movie features (first 10): [0.9630576  0.9218111  0.95030344 0.8993994  0.6391391  0.
 0.         0.         0.         0.        ]


In [76]:
# Get a historical row for testing value signals
sample_historical = historical.iloc[0]

print(f"Sample historical broadcast:")
print(f"  Date: {sample_historical['date']}")
print(f"  Channel: {sample_historical['channel']}")
print(f"  Catalog ID: {sample_historical['catalog_id']}")

Sample historical broadcast:
  Date: 2024-01-01 00:00:00
  Channel: M6_T_PL
  Catalog ID: <NA>


In [77]:
# Compute value signals for this historical row
# Note: We'll need to check the exact method signature in reward.py
# This is a placeholder - we'll update once we verify the API

print("Value signals computation to be verified in next phase...")
print("We need to check RewardCalculator API for computing signals on historical data.")

Value signals computation to be verified in next phase...
We need to check RewardCalculator API for computing signals on historical data.


## 7. Explore Available Movies

In [78]:
# Get available movies for a sample date
sample_date = datetime(2024, 3, 2)  # Use datetime, not date
env.get_available_movies(sample_date)  # This updates env.available_movies

print(f"Available movies on {sample_date.date()}: {len(env.available_movies)}")
print(f"Sample available catalog IDs: {env.available_movies[:10]}")

Available movies on 2024-03-02: 1556
Sample available catalog IDs: ['348', '9799', '1637', '1639', '9426', '2770', '2501', '49040', '436167', '544']


## 8. Summary Statistics

In [79]:
print("=" * 60)
print("SETUP VALIDATION SUMMARY")
print("=" * 60)
print(f"")
print(f"✅ Imports: All successful")
print(f"✅ Catalog: {len(catalog)} movies loaded")
print(f"✅ Historical: {len(historical)} broadcasts loaded")
print(f"✅ Date range: {historical['date'].min()} to {historical['date'].max()}")
print(f"✅ Channels: {', '.join(historical['channel'].unique())}")
print(f"✅ Environment: Initialized")
print(f"    - Catalog size: {len(env.catalog_df)}")
print(f"    - Memory size: {env.memory_size}")
print(f"    - Context features: 18D (cached: {len(env.context_features_cache)})")
print(f"✅ Reward Calculator: Initialized via environment")
print(f"✅ Audience Model: Loaded")
print(f"")
print("=" * 60)
print("Ready to proceed to Phase 1: Policy Interface & Simple Baselines")
print("=" * 60)

SETUP VALIDATION SUMMARY

✅ Imports: All successful
✅ Catalog: 13682 movies loaded
✅ Historical: 3338 broadcasts loaded
✅ Date range: 2024-01-01 00:00:00 to 2025-02-22 00:00:00
✅ Channels: M6_T_PL, France 3, TF1_T_PL, RTS 1, RTS 2, France 2
✅ Environment: Initialized
    - Catalog size: 13682
    - Memory size: 100
    - Context features: 18D (cached: 1)
✅ Reward Calculator: Initialized via environment
✅ Audience Model: Loaded

Ready to proceed to Phase 1: Policy Interface & Simple Baselines
